> **STAGE-2 SUPERSEDED NOTICE:** See **[`08_confirmatory_operating_point_matching.ipynb`](file://./notebooks/08_confirmatory_operating_point_matching.ipynb)** for the Second-Stage practical equivalence margin audit, exact non-central-$t$ CIs, LOSO sensitivity, 1D-for-2D root-finding proof, and the 30-run fresh-seed (`201..205`) confirmatory design.

# Notebook 01 — Phase 2: Operating-Point Matching, Formal TOST Equivalence, & Continuous Root-Finding

**Purpose:**
1. Derive scientifically justified equivalence margins ($\delta_{\text{PPL}}, \delta_{H}$) independently of observed pairwise treatment effects using:
   - **Rationale 1 (Monte Carlo Evaluation Noise Floor):** Finite-sample bootstrap uncertainty at $M=1{,}000$ sequences per run across 48 non-collapsed `ELF-B` runs.
   - **Rationale 2 (Cross-Seed Repeatability Limit):** Pooled within-cell cross-seed standard deviation $\sigma_{\text{seed}}$ across all replicated configurations ($df=50$ on `ELF-B`, $df=18$ on `ELF-M`).
2. Evaluate all 6 matched operating-point pairs (`M1`, `S1`, `S2`, `S3`, `Refined-1`, `Refined-2`) using exact **Satterthwaite Welch $t$-tests**, **Two One-Sided Tests (TOST)** at Tier I ($\pm 2.5\%$ Gen-PPL, $\pm 0.015$ nats $H_{\text{ps}}$) and Tier II ($\pm 5.0\%$ Gen-PPL, $\pm 0.025$ nats $H_{\text{ps}}$), $95\%$ and $90\%$ CIs, **Cohen's $d$**, and small-sample bias-corrected **Hedges' $g$**.
3. Perform **matching-tolerance sensitivity analysis** across candidate tolerances.
4. Solve for **continuous root-matched $\gamma^*$ operating points** on `ELF-B` and `ELF-M` where predicted $|\Delta\text{Gen-PPL}| < 0.3\%$ and $|\Delta H_{\text{ps}}| < 0.003$ nats for confirmatory GPU execution in `Notebook 02`.

In [1]:
import json
import math
from pathlib import Path
import numpy as np

# Locate project root (works both locally and in Google Colab)
ROOT = Path.cwd()
if not (ROOT / "analysis" / "out" / "critical_foundation_stats.json").exists():
    if (ROOT.parent / "analysis" / "out" / "critical_foundation_stats.json").exists():
        ROOT = ROOT.parent

stats_path = ROOT / "analysis" / "out" / "critical_foundation_stats.json"
data = json.loads(stats_path.read_text(encoding="utf-8"))
p2 = data["phase2_equivalence_and_matching"]

print("=== 1. SCIENTIFIC EQUIVALENCE MARGIN DERIVATIONS ===")
r1 = p2["scientific_margin_derivations"]["rationale_1_monte_carlo_measurement_noise"]
r2 = p2["scientific_margin_derivations"]["rationale_2_within_cell_cross_seed_repeatability"]
print(f"Rationale 1 (Monte Carlo M=1,000 95% CI half-width): median = {r1['median_95ci_rel_half_width_pct']:.2f}%, 2-sample noise floor = {r1['two_sample_mc_95_bound_pct']:.2f}%")
print(f"Rationale 2 (Cross-Seed Repeatability, ELF-B df=50): sigma_seed(Gen-PPL) = {r2['elfb_pooled_seed_sd_rel_pct']:.2f}% (95% limit = {r2['elfb_95_repeatability_rel_pct']:.2f}%), sigma_seed(H_ps) = {r2['elfb_pooled_seed_sd_hps_nats']:.4f} nats")
print(f"Rationale 2 (Cross-Seed Repeatability, ELF-M df=18): sigma_seed(Gen-PPL) = {r2['elfm_pooled_seed_sd_rel_pct']:.2f}%, sigma_seed(H_ps) = {r2['elfm_pooled_seed_sd_hps_nats']:.4f} nats")

=== 1. SCIENTIFIC EQUIVALENCE MARGIN DERIVATIONS ===
Rationale 1 (Monte Carlo M=1,000 95% CI half-width): median = 1.94%, 2-sample noise floor = 2.74%
Rationale 2 (Cross-Seed Repeatability, ELF-B df=50): sigma_seed(Gen-PPL) = 1.11% (95% limit = 3.12%), sigma_seed(H_ps) = 0.0029 nats
Rationale 2 (Cross-Seed Repeatability, ELF-M df=18): sigma_seed(Gen-PPL) = 0.95%, sigma_seed(H_ps) = 0.0054 nats


In [1]:
print("=== 2. FORMAL TOST EQUIVALENCE & EFFECT SIZE AUDIT ACROSS EXISTING PAIRS ===")
header = f"{'Pair':<18} {'Seeds':<7} {'dPPL(%)':>8} {'90% CI dPPL':>18} {'TOST_2.5%':>10} {'TOST_5%':>9} {'dH(nats)':>10} {'TOST_015':>9} {'TOST_025':>9}"
print(header)
print("-" * len(header))
for name, rep in p2["pair_tost_and_effect_size_reports"].items():
    gp1 = rep["gen_ppl_tier1_2.5pct"]
    gp2 = rep["gen_ppl_tier2_5.0pct"]
    hp1 = rep["h_ps_tier1_0.015nats"]
    hp2 = rep["h_ps_tier2_0.025nats"]
    ci90_str = f"[{gp1['ci90_tost'][0]:+.2f}, {gp1['ci90_tost'][1]:+.2f}]"
    seeds_str = f"{gp1['n_A']}v{gp1['n_B']}"
    print(f"{name:<18} {seeds_str:<7} {gp1['rel_diff_pct_vs_A']:>+7.2f}% {ci90_str:>18} "
          f"{str(gp1['tost_equivalent_at_05']):>10} {str(gp2['tost_equivalent_at_05']):>9} "
          f"{hp1['diff_B_minus_A']:>+10.4f} {str(hp1['tost_equivalent_at_05']):>9} {str(hp2['tost_equivalent_at_05']):>9}")

=== 2. FORMAL TOST EQUIVALENCE & EFFECT SIZE AUDIT ACROSS EXISTING PAIRS ===
Pair               Seeds    dPPL(%)        90% CI dPPL  TOST_2.5%   TOST_5%   dH(nats)  TOST_015  TOST_025
----------------------------------------------------------------------------------------------------------
M1_ELFM_5seed      5v5       +0.41%     [-0.16, +0.35]       True      True    -0.0210     False     False
S1_ELFB_5seed      5v5       -0.96%     [-0.59, +0.14]      False      True    +0.0044      True      True
S2_ELFB_2seed      2v2       -2.10%     [-2.64, +1.40]      False     False    -0.0079     False     False
S3_ELFB_2seed      2v2       +4.03%     [-0.19, +2.80]      False     False    -0.0097     False      True
Refined1_ELFB_5seed 5v5       +4.31%     [+1.15, +1.81]      False     False    -0.0047      True      True
Refined2_ELFB_5seed 5v5       +5.14%     [+0.99, +1.71]      False     False    +0.0123     False      True


In [1]:
print("=== 3. STANDARDIZED EFFECT SIZES (HEDGES' g & 95% WELCH CIs) ON OUTCOME METRICS ===")
metrics_order = ["rep4", "within_distinct2", "h_pool", "jsd_inter_sample", "distinct2", "self_bleu", "jcd_bits_per_byte", "mauve"]
for name, rep in p2["pair_tost_and_effect_size_reports"].items():
    print(f"\n--- {name} (Cell A: SC={rep['cell_A']['sc_cfg']}, g={rep['cell_A']['gamma']} vs Cell B: SC={rep['cell_B']['sc_cfg']}, g={rep['cell_B']['gamma']}) ---")
    print(f"{'Metric':<20} {'Mean A':>10} {'Mean B':>10} {'Diff (B-A)':>12} {'Rel %':>9} {'df_Satt':>8} {'Hedges g':>10} {'p_Welch':>12}")
    for m in metrics_order:
        st = rep["secondary_metrics_hedges_g_and_welch"][m]
        print(f"{m:<20} {st['mean_A']:>10.4f} {st['mean_B']:>10.4f} {st['diff_B_minus_A']:>+12.4f} "
              f"{st['rel_diff_pct_vs_A']:>+8.2f}% {st['df_satt']:>8.2f} {st['hedges_g']:>+10.2f} {st['p_welch_2sided']:>12.2e}")

=== 3. STANDARDIZED EFFECT SIZES (HEDGES' g & 95% WELCH CIs) ON OUTCOME METRICS ===

--- M1_ELFM_5seed (Cell A: SC=1.0, g=2.0 vs Cell B: SC=4.0, g=0.5) ---
Metric                   Mean A     Mean B   Diff (B-A)     Rel %  df_Satt   Hedges g      p_Welch
rep4                     0.0297     0.0644      +0.0347  +116.86%     5.86     +30.63     4.15e-09
within_distinct2         0.7987     0.7549      -0.0439    -5.49%     7.29     -19.00     3.13e-09
h_pool                   6.8869     7.1427      +0.2558    +3.71%     7.40     +17.69     4.21e-09
jsd_inter_sample         1.6774     1.9548      +0.2774   +16.54%     6.26     +26.08     3.87e-09
distinct2                0.2856     0.3208      +0.0352   +12.34%     6.66     +14.77     6.27e-08
self_bleu               45.3643    38.9484      -6.4159   -14.14%     7.88     -15.21     5.27e-09
jcd_bits_per_byte        0.0856     0.0733      -0.0123   -14.33%     6.25      -5.42     6.14e-05
mauve                    0.3478     0.5308      +0.1

In [1]:
print("=== 4. CONTINUOUS ROOT-FINDING CANDIDATES FOR EXACT OPERATING-POINT MATCHING (Colab Notebook 02) ===")
print(f"{'Model':<7} {'Target Cell B (SC, g)':<22} {'Root g* on SC=1':<17} {'Pred dPPL(%)':>13} {'Pred dH(nats)':>15}")
print("-" * 78)
for cand in p2["root_finding_new_candidate_pairs"]:
    tb = cand["target_B"]
    jm = cand["joint_pareto_match_on_SC1"]
    if abs(jm["pred_rel_ppl_diff_pct"]) <= 1.0 and abs(jm["pred_h_diff_nats"]) <= 0.015:
        tb_str = f"SC={tb['sc_cfg']}, g={tb['gamma']}"
        print(f"{cand['model']:<7} {tb_str:<22} {jm['gamma_opt']:<17.3f} {jm['pred_rel_ppl_diff_pct']:>+12.2f}% {jm['pred_h_diff_nats']:>+15.4f}")

=== 4. CONTINUOUS ROOT-FINDING CANDIDATES FOR EXACT OPERATING-POINT MATCHING (Colab Notebook 02) ===
Model   Target Cell B (SC, g)  Root g* on SC=1    Pred dPPL(%)   Pred dH(nats)
------------------------------------------------------------------------------
ELF-B   SC=2.0, g=0.25         0.781                    +0.58%         -0.0083
ELF-B   SC=2.0, g=0.5          0.947                    +0.39%         -0.0066
ELF-B   SC=2.0, g=1.0          1.349                    -0.04%         +0.0002
ELF-B   SC=2.0, g=1.5          1.766                    -0.57%         +0.0050
ELF-B   SC=3.0, g=0.5          1.202                    +0.91%         -0.0128
ELF-B   SC=3.0, g=1.0          1.548                    +0.03%         -0.0005
ELF-B   SC=3.0, g=1.5          1.930                    -0.91%         +0.0082
ELF-B   SC=4.0, g=1.0          1.652                    +0.26%         -0.0028
ELF-B   SC=4.0, g=1.5          2.001                    -0.93%         +0.0046
